# Statistical Arbitrage with Pairs Trading

This notebook demonstrates a pairs trading strategy, a common form of statistical arbitrage in quantitative finance. Pairs trading seeks to exploit temporary divergences in the prices of historically correlated assets.

Key concepts covered:
1. Cointegration testing between S&P 500 components
2. Spread calculation and z-score normalization
3. Trading signal generation
4. Strategy performance evaluation

## Setup and Data Collection

First, let's import the necessary libraries and fetch the price data for potential pairs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
import yfinance as yf

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

Let's collect stock data for several technology companies in the S&P 500 that might exhibit cointegration.

In [ ]:
# Define a list of tech stocks from S&P 500
tech_stocks = ['MSFT', 'AAPL', 'GOOGL', 'META', 'AMZN', 'NVDA', 'AMD', 'INTC', 'IBM', 'CSCO']

# Fetch historical data (last 2 years)
stock_data = yf.download(tech_stocks, start='2023-01-01', end=None)

# Extract adjusted close prices
prices = stock_data['Adj Close']

# Display the first few rows
print(f"Data period: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Number of trading days: {len(prices)}")
prices.head()

Let's visualize the price movements of these stocks to get a feel for their relationship.

In [ ]:
# Normalize prices to start from 100 for better comparison
normalized_prices = prices.div(prices.iloc[0]).mul(100)

# Plot normalized prices
plt.figure(figsize=(14, 8))
for col in normalized_prices.columns:
    plt.plot(normalized_prices.index, normalized_prices[col], label=col)
plt.title('Normalized Stock Prices (Base = 100)')
plt.xlabel('Date')
plt.ylabel('Normalized Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Finding Cointegrated Pairs

Cointegration is a statistical property of time series that indicates they share a long-term equilibrium relationship, even if both series are non-stationary. This is crucial for pairs trading.

Let's test all possible pairs of stocks for cointegration:

In [ ]:
# Function to test cointegration between two stocks
def test_cointegration(stock1, stock2, data, significance=0.05):
    # Get the price series
    stock1_prices = data[stock1].dropna()
    stock2_prices = data[stock2].dropna()
    
    # Ensure both series have the same length
    common_dates = stock1_prices.index.intersection(stock2_prices.index)
    stock1_prices = stock1_prices.loc[common_dates]
    stock2_prices = stock2_prices.loc[common_dates]
    
    # Test for cointegration
    score, pvalue, _ = coint(stock1_prices, stock2_prices)
    
    # Return the results
    return {
        'stock1': stock1,
        'stock2': stock2,
        'p-value': pvalue,
        'cointegrated': pvalue < significance
    }

In [ ]:
# Test all possible pairs
results = []
for i, stock1 in enumerate(tech_stocks):
    for stock2 in tech_stocks[i+1:]:
        result = test_cointegration(stock1, stock2, prices)
        results.append(result)

# Convert results to DataFrame and sort by p-value
results_df = pd.DataFrame(results).sort_values('p-value')
results_df

Let's identify the most promising pair(s) based on the cointegration test results and visualize them:

In [ ]:
# Filter for cointegrated pairs
cointegrated_pairs = results_df[results_df['cointegrated']]
print(f"Found {len(cointegrated_pairs)} cointegrated pairs at 5% significance level")
cointegrated_pairs

In [ ]:
# Select the pair with the lowest p-value for further analysis
# (if there are any cointegrated pairs)
if not cointegrated_pairs.empty:
    best_pair = cointegrated_pairs.iloc[0]
    stock1 = best_pair['stock1']
    stock2 = best_pair['stock2']
    print(f"Selected pair for analysis: {stock1} and {stock2} with p-value: {best_pair['p-value']:.6f}")
else:
    # If no cointegrated pairs found, just pick the pair with lowest p-value
    best_pair = results_df.iloc[0]
    stock1 = best_pair['stock1']
    stock2 = best_pair['stock2']
    print(f"No cointegrated pairs at 5% threshold. Using {stock1} and {stock2} with lowest p-value: {best_pair['p-value']:.6f}")

Let's visualize the selected pair and calculate the hedge ratio using OLS regression:

In [ ]:
# Extract the price series for the selected pair
stock1_prices = prices[stock1]
stock2_prices = prices[stock2]

# Plot the normalized prices
plt.figure(figsize=(14, 7))
plt.plot(stock1_prices / stock1_prices.iloc[0], label=stock1)
plt.plot(stock2_prices / stock2_prices.iloc[0], label=stock2)
plt.title(f'Normalized Prices: {stock1} vs {stock2}')
plt.xlabel('Date')
plt.ylabel('Normalized Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the hedge ratio using OLS regression
model = OLS(stock1_prices, stock2_prices)
result = model.fit()
hedge_ratio = result.params[0]

print(f"Hedge Ratio: {hedge_ratio:.4f}")
print(f"This means we should buy 1 share of {stock1} and sell {hedge_ratio:.4f} shares of {stock2}")
print("\nRegression Summary:")
print(result.summary())

## Calculating the Spread and Z-Score

The spread represents the difference between the two assets after accounting for the hedge ratio. In pairs trading, we track the z-score of this spread to generate trading signals.

In [ ]:
# Calculate the spread
spread = stock1_prices - hedge_ratio * stock2_prices

# Plot the spread
plt.figure(figsize=(14, 7))
plt.plot(spread)
plt.axhline(y=spread.mean(), color='r', linestyle='--', label='Mean')
plt.title(f'Spread between {stock1} and {stock2}')
plt.xlabel('Date')
plt.ylabel('Spread')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Test if the spread is stationary using the Augmented Dickey-Fuller test
adf_result = adfuller(spread.dropna())
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print("Critical Values:")
for key, value in adf_result[4].items():
    print(f"\t{key}: {value:.4f}")

# Interpret the result
if adf_result[1] < 0.05:
    print("\nThe spread is stationary at the 5% significance level")
else:
    print("\nThe spread is not stationary at the 5% significance level")

In [ ]:
# Calculate the z-score of the spread
def calculate_zscore(spread, window=20):
    """Calculate the z-score of a spread over a rolling window"""
    rolling_mean = spread.rolling(window=window).mean()
    rolling_std = spread.rolling(window=window).std()
    z_score = (spread - rolling_mean) / rolling_std
    return z_score

z_score = calculate_zscore(spread)

# Plot the z-score
plt.figure(figsize=(14, 7))
plt.plot(z_score)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='green', linestyle='--', alpha=0.5)
plt.axhline(y=-1, color='green', linestyle='--', alpha=0.5)
plt.axhline(y=2, color='red', linestyle='--', alpha=0.5)
plt.axhline(y=-2, color='red', linestyle='--', alpha=0.5)
plt.title(f'Z-Score of Spread between {stock1} and {stock2}')
plt.xlabel('Date')
plt.ylabel('Z-Score')
plt.grid(True)
plt.tight_layout()
plt.show()

## Generating Trading Signals

Now, let's generate trading signals based on the z-score and implement a pairs trading strategy.

Trading rules:
1. When z-score < -2: The spread is too low, so we buy the spread (buy stock1, sell stock2)
2. When z-score > 2: The spread is too high, so we sell the spread (sell stock1, buy stock2)
3. When -0.5 < z-score < 0.5: Close any existing positions (the spread has mean-reverted)

In [ ]:
# Generate trading signals
def generate_signals(z_score, entry_threshold=2, exit_threshold=0.5):
    """Generate trading signals based on z-score thresholds"""
    signals = pd.DataFrame(index=z_score.index)
    signals['z_score'] = z_score
    
    # Initialize signal column
    signals['signal'] = 0
    
    # Buy signal when z-score is below negative threshold
    signals.loc[signals['z_score'] < -entry_threshold, 'signal'] = 1
    
    # Sell signal when z-score is above positive threshold
    signals.loc[signals['z_score'] > entry_threshold, 'signal'] = -1
    
    # Exit signal when z-score is between -exit_threshold and exit_threshold
    signals.loc[(signals['z_score'] > -exit_threshold) & 
                (signals['z_score'] < exit_threshold), 'signal'] = 0
    
    return signals

# Generate signals
signals = generate_signals(z_score)

# Plot signals
plt.figure(figsize=(14, 10))

# Top panel: Z-score
plt.subplot(2, 1, 1)
plt.plot(z_score)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axhline(y=2, color='red', linestyle='--', alpha=0.5)
plt.axhline(y=-2, color='red', linestyle='--', alpha=0.5)
plt.axhline(y=0.5, color='green', linestyle='--', alpha=0.5)
plt.axhline(y=-0.5, color='green', linestyle='--', alpha=0.5)
plt.title(f'Z-Score and Trading Signals')
plt.ylabel('Z-Score')
plt.grid(True)

# Bottom panel: Trading signals
plt.subplot(2, 1, 2)
plt.plot(signals['signal'])
plt.title('Trading Signals (1: Buy Spread, -1: Sell Spread, 0: No Position)')
plt.xlabel('Date')
plt.ylabel('Signal')
plt.grid(True)
plt.tight_layout()
plt.show()

## Backtesting the Strategy

Let's implement and backtest our pairs trading strategy.

In [ ]:
# Implement the trading strategy
def backtest_strategy(signals, stock1_prices, stock2_prices, hedge_ratio):
    """Backtest a pairs trading strategy"""
    # Create a copy of signals
    positions = signals.copy()
    
    # Generate positions (shifted by 1 day to avoid look-ahead bias)
    positions['signal'] = positions['signal'].shift(1).fillna(0)
    
    # Calculate returns for each leg of the trade
    positions['stock1_returns'] = stock1_prices.pct_change()
    positions['stock2_returns'] = stock2_prices.pct_change()
    
    # Calculate strategy returns
    # When signal = 1: Long stock1, Short stock2
    # When signal = -1: Short stock1, Long stock2
    positions['strategy_returns'] = positions['signal'] * (
        positions['stock1_returns'] - hedge_ratio * positions['stock2_returns']
    )
    
    # Calculate cumulative returns
    positions['cum_strategy_returns'] = (1 + positions['strategy_returns']).cumprod() - 1
    positions['cum_stock1_returns'] = (1 + positions['stock1_returns']).cumprod() - 1
    positions['cum_stock2_returns'] = (1 + positions['stock2_returns']).cumprod() - 1
    
    return positions

# Backtest the strategy
backtest_results = backtest_strategy(signals, stock1_prices, stock2_prices, hedge_ratio)

# Plot cumulative returns
plt.figure(figsize=(14, 7))
plt.plot(backtest_results['cum_strategy_returns'], label='Pairs Strategy')
plt.plot(backtest_results['cum_stock1_returns'], label=stock1)
plt.plot(backtest_results['cum_stock2_returns'], label=stock2)
plt.title('Cumulative Returns Comparison')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Performance Analysis

Let's analyze the performance of our pairs trading strategy.

In [ ]:
# Calculate strategy performance metrics
def calculate_performance_metrics(returns):
    """Calculate performance metrics for a returns series"""
    # Drop NaN values
    returns = returns.dropna()
    
    # Calculate metrics
    total_return = (1 + returns).cumprod().iloc[-1] - 1
    annualized_return = (1 + total_return) ** (252 / len(returns)) - 1
    annualized_volatility = returns.std() * np.sqrt(252)
    sharpe_ratio = annualized_return / annualized_volatility if annualized_volatility != 0 else 0
    max_drawdown = (1 - (1 + returns).cumprod() / (1 + returns).cumprod().cummax()).max()
    win_rate = len(returns[returns > 0]) / len(returns)
    
    # Return metrics
    return {
        'Total Return': total_return,
        'Annualized Return': annualized_return,
        'Annualized Volatility': annualized_volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Max Drawdown': max_drawdown,
        'Win Rate': win_rate
    }

In [ ]:
# Calculate performance metrics for strategy and individual stocks
strategy_metrics = calculate_performance_metrics(backtest_results['strategy_returns'])
stock1_metrics = calculate_performance_metrics(backtest_results['stock1_returns'])
stock2_metrics = calculate_performance_metrics(backtest_results['stock2_returns'])

# Create a DataFrame to compare metrics
performance_comparison = pd.DataFrame({
    'Pairs Strategy': strategy_metrics,
    stock1: stock1_metrics,
    stock2: stock2_metrics
})

# Format the percentages
for metric in ['Total Return', 'Annualized Return', 'Annualized Volatility', 'Max Drawdown', 'Win Rate']:
    performance_comparison.loc[metric] = performance_comparison.loc[metric].apply(lambda x: f"{x*100:.2f}%")

# Format the Sharpe Ratio
performance_comparison.loc['Sharpe Ratio'] = performance_comparison.loc['Sharpe Ratio'].apply(lambda x: f"{float(x):.2f}")

# Display performance comparison
print("Performance Comparison:")
display(performance_comparison)

## Trade Analysis

Let's analyze the individual trades generated by our strategy.

In [ ]:
# Identify trade entry and exit points
def analyze_trades(positions):
    """Identify and analyze individual trades"""
    # Create a trades DataFrame
    trades = []
    
    # Identify changes in position (trade entry/exit)
    position_changes = positions['signal'].diff().fillna(0)
    
    # Find all non-zero position changes
    trade_days = position_changes[position_changes != 0].index
    
    current_position = 0
    entry_date = None
    entry_zscore = None
    
    for day in trade_days:
        change = position_changes.loc[day]
        new_position = positions['signal'].loc[day]
        
        # If we're entering a new position
        if current_position == 0 and new_position != 0:
            entry_date = day
            entry_zscore = positions['z_score'].loc[day]
            current_position = new_position
        
        # If we're closing a position
        elif current_position != 0 and (new_position == 0 or new_position == -current_position):
            exit_date = day
            exit_zscore = positions['z_score'].loc[day]
            
            # Calculate trade return
            if current_position == 1:  # Long the spread
                trade_type = "Long Spread"
                entry_action = f"Buy {stock1}, Sell {stock2}"
                exit_action = f"Sell {stock1}, Buy {stock2}"
            else:  # Short the spread
                trade_type = "Short Spread"
                entry_action = f"Sell {stock1}, Buy {stock2}"
                exit_action = f"Buy {stock1}, Sell {stock2}"
                
            # Calculate trade return
            if entry_date and exit_date:
                trade_return = positions.loc[entry_date:exit_date, 'strategy_returns'].sum()
                trade_duration = len(positions.loc[entry_date:exit_date])
                
                trades.append({
                    'Entry Date': entry_date,
                    'Exit Date': exit_date,
                    'Trade Type': trade_type,
                    'Entry Action': entry_action,
                    'Exit Action': exit_action,
                    'Entry Z-Score': entry_zscore,
                    'Exit Z-Score': exit_zscore,
                    'Trade Return': trade_return,
                    'Duration (Days)': trade_duration
                })
            
            # Reset for next trade
            entry_date = None if new_position == 0 else day
            entry_zscore = None if new_position == 0 else positions['z_score'].loc[day]
            current_position = new_position
    
    # If we still have an open position at the end
    if entry_date and current_position != 0:
        exit_date = positions.index[-1]
        exit_zscore = positions['z_score'].iloc[-1]
        
        if current_position == 1:  # Long the spread
            trade_type = "Long Spread (Open)"
            entry_action = f"Buy {stock1}, Sell {stock2}"
            exit_action = "Open Position"
        else:  # Short the spread
            trade_type = "Short Spread (Open)"
            entry_action = f"Sell {stock1}, Buy {stock2}"
            exit_action = "Open Position"
            
        trade_return = positions.loc[entry_date:exit_date, 'strategy_returns'].sum()
        trade_duration = len(positions.loc[entry_date:exit_date])
        
        trades.append({
            'Entry Date': entry_date,
            'Exit Date': exit_date,
            'Trade Type': trade_type,
            'Entry Action': entry_action,
            'Exit Action': exit_action,
            'Entry Z-Score': entry_zscore,
            'Exit Z-Score': exit_zscore,
            'Trade Return': trade_return,
            'Duration (Days)': trade_duration
        })
    
    return pd.DataFrame(trades)

# Analyze trades
trades_analysis = analyze_trades(backtest_results)

# Format trade returns as percentages
trades_analysis['Trade Return'] = trades_analysis['Trade Return'].apply(lambda x: f"{x*100:.2f}%")

# Display trade analysis
print(f"Total Trades: {len(trades_analysis)}")
display(trades_analysis.sort_values('Entry Date'))

## Improving the Strategy

Here are some ways to potentially improve this pairs trading strategy:

1. **Dynamic Z-Score Thresholds**: Adjust entry/exit thresholds based on recent volatility
2. **Stop-Loss Implementation**: Add stop-loss rules to limit downside risk
3. **Multiple Pairs**: Trade a portfolio of cointegrated pairs for diversification
4. **Rebalancing**: Periodically reassess cointegration and hedge ratios
5. **Transaction Costs**: Incorporate trading fees and slippage for more realistic results
6. **Alternative Spread Calculation**: Experiment with different methods for calculating the spread
7. **Machine Learning**: Use ML to predict spread movements or optimize parameters

## Conclusion

Statistical arbitrage through pairs trading offers a market-neutral approach to algorithmic trading. By identifying cointegrated securities and trading the mean-reverting spread between them, traders can potentially generate profits regardless of overall market direction.

Key benefits of this approach include:
- Reduced directional market risk
- Mathematically rigorous entry/exit signals
- Adaptability across different market sectors and conditions

However, challenges remain, including:
- Cointegration relationships can break down
- Execution costs can significantly impact profitability
- Requires constant monitoring and rebalancing

For S&P 500 components, pairs trading can be particularly effective when applied to stocks within the same sector, as they often share common risk factors and tend to move together over the long term.